# 06 — DistilBERT Sentiment Classification

In this notebook, we build a Transformer-based sentiment classifier
for the IMDb dataset.

Before training, we first learn how Transformer inputs work:

1. Tokenization
2. Token IDs
3. Special tokens
4. Padding
5. Truncation
6. Attention masks

Then we fine-tune DistilBERT for binary sentiment classification.

Labels:

- 0 = negative
- 1 = positive

In [3]:
%pip install transformers

Looking in indexes: https://pypi.org/simple/
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached markdown_it_py-4.2.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ---------------------------------------- 0.0/11.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.6 MB ? eta -:--:--
    --------------------------------------- 0.3/11.6 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.6 MB 1.3 MB/s eta 0:00:09
   -- ------------------------------------- 0.8/11.6 MB 1.3 MB/s eta 0:00:09
   ---- ----------------------------------- 1.3/11.6 MB 1.7 MB/s eta 0:00:07
   ----- ---------------------------------- 1.6/11.6 MB 1.6 MB/s eta 0:00:07
   ------ ---------------------


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from __future__ import annotations

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch

from datasets import load_from_disk

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)

In [5]:
CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )

print(
    "Project root:",
    PROJECT_ROOT,
)

Project root: D:\app\ai\Sentiment Analysis Traditional ML vs Transformers


In [6]:
DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "imdb_clean_splits"
)

In [7]:
dataset = load_from_disk(
    str(DATA_PATH)
)

train_df = (
    dataset["train"]
    .to_pandas()[["text", "label"]]
    .copy()
)

validation_df = (
    dataset["validation"]
    .to_pandas()[["text", "label"]]
    .copy()
)

print(
    "Train:",
    f"{len(train_df):,}",
)

print(
    "Validation:",
    f"{len(validation_df):,}",
)

Train: 19,823
Validation: 4,956


In [8]:
MODEL_NAME = (
    "distilbert-base-uncased"
)

In [9]:
tokenizer = (
    AutoTokenizer
    .from_pretrained(
        MODEL_NAME
    )
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

d:\app\ai\Sentiment Analysis Traditional ML vs Transformers\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mmnst\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [10]:
example_text = (
    "I did not like this movie."
)

print(
    example_text
)

I did not like this movie.


In [11]:
tokens = tokenizer.tokenize(
    example_text
)

tokens

['i', 'did', 'not', 'like', 'this', 'movie', '.']

In [12]:
token_ids = (
    tokenizer
    .convert_tokens_to_ids(
        tokens
    )
)

list(
    zip(
        tokens,
        token_ids,
    )
)

[('i', 1045),
 ('did', 2106),
 ('not', 2025),
 ('like', 2066),
 ('this', 2023),
 ('movie', 3185),
 ('.', 1012)]

In [13]:
encoded_example = tokenizer(
    example_text
)

encoded_example

{'input_ids': [101, 1045, 2106, 2025, 2066, 2023, 3185, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [14]:
tokens_with_special_tokens = (
    tokenizer
    .convert_ids_to_tokens(
        encoded_example[
            "input_ids"
        ]
    )
)

tokens_with_special_tokens

['[CLS]', 'i', 'did', 'not', 'like', 'this', 'movie', '.', '[SEP]']

In [15]:
token_table = pd.DataFrame(
    {
        "token": (
            tokens_with_special_tokens
        ),
        "input_id": (
            encoded_example[
                "input_ids"
            ]
        ),
        "attention_mask": (
            encoded_example[
                "attention_mask"
            ]
        ),
    }
)

token_table

,token,input_id,attention_mask
0,[CLS],101,1
1,i,1045,1
2,did,2106,1
3,not,2025,1
4,like,2066,1
5,this,2023,1
6,movie,3185,1
7,.,1012,1
8,[SEP],102,1


## 2. Padding and attention masks

Transformer batches must have a rectangular shape.

Because texts have different lengths, shorter sequences need padding.

The attention mask tells the model which positions contain real tokens
and which positions are only padding.

In [16]:
batch_texts = [
    "Great movie.",
    "This movie was surprisingly good and very entertaining.",
]

batch_texts

['Great movie.', 'This movie was surprisingly good and very entertaining.']

In [17]:
without_padding = tokenizer(
    batch_texts,
    padding=False,
)

without_padding

{'input_ids': [[101, 2307, 3185, 1012, 102], [101, 2023, 3185, 2001, 10889, 2204, 1998, 2200, 14036, 1012, 102]], 'token_type_ids': [[0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}

In [18]:
for index, input_ids in enumerate(
    without_padding["input_ids"]
):
    print(
        f"Text {index}:",
        len(input_ids),
        "tokens",
    )

Text 0: 5 tokens
Text 1: 11 tokens


In [19]:
padded_batch = tokenizer(
    batch_texts,
    padding=True,
)

padded_batch

{'input_ids': [[101, 2307, 3185, 1012, 102, 0, 0, 0, 0, 0, 0], [101, 2023, 3185, 2001, 10889, 2204, 1998, 2200, 14036, 1012, 102]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}

In [20]:
for index, input_ids in enumerate(
    padded_batch["input_ids"]
):
    print(
        f"Text {index}:",
        len(input_ids),
        "tokens",
    )

Text 0: 11 tokens
Text 1: 11 tokens


In [21]:
for index, input_ids in enumerate(
    padded_batch["input_ids"]
):
    tokens = tokenizer.convert_ids_to_tokens(
        input_ids
    )

    print(
        f"Text {index}:"
    )

    print(tokens)

    print()

Text 0:
['[CLS]', 'great', 'movie', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']

Text 1:
['[CLS]', 'this', 'movie', 'was', 'surprisingly', 'good', 'and', 'very', 'entertaining', '.', '[SEP]']



In [22]:
for index in range(
    len(batch_texts)
):
    tokens = (
        tokenizer
        .convert_ids_to_tokens(
            padded_batch[
                "input_ids"
            ][index]
        )
    )

    attention_mask = (
        padded_batch[
            "attention_mask"
        ][index]
    )

    display_df = pd.DataFrame(
        {
            "token": tokens,
            "attention_mask": attention_mask,
        }
    )

    print(
        f"Text {index}:"
    )

    display(display_df)

Text 0:


,token,attention_mask
0,[CLS],1
1,great,1
2,movie,1
3,.,1
4,[SEP],1
5,[PAD],0
6,[PAD],0
7,[PAD],0
8,[PAD],0
9,[PAD],0


Text 1:


,token,attention_mask
0,[CLS],1
1,this,1
2,movie,1
3,was,1
4,surprisingly,1
5,good,1
6,and,1
7,very,1
8,entertaining,1
9,.,1


In [23]:
tensor_batch = tokenizer(
    batch_texts,
    padding=True,
    return_tensors="pt",
)

tensor_batch

{'input_ids': tensor([[  101,  2307,  3185,  1012,   102,     0,     0,     0,     0,     0,
             0],
        [  101,  2023,  3185,  2001, 10889,  2204,  1998,  2200, 14036,  1012,
           102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [24]:
print(
    type(
        tensor_batch[
            "input_ids"
        ]
    )
)

print(
    type(
        tensor_batch[
            "attention_mask"
        ]
    )
)

<class 'torch.Tensor'>
<class 'torch.Tensor'>


In [25]:
long_text = (
    "This movie was very interesting. "
    * 200
)

print(
    "Characters:",
    len(long_text),
)

Characters: 6600


In [26]:
long_tokens = tokenizer.tokenize(
    long_text
)

print(
    "Number of tokens:",
    len(long_tokens),
)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1200 > 512). Running this sequence through the model will result in indexing errors


Number of tokens: 1200


In [27]:
truncated_example = tokenizer(
    long_text,
    truncation=True,
    max_length=16,
)

print(
    "Number of input IDs:",
    len(
        truncated_example[
            "input_ids"
        ]
    ),
)

Number of input IDs: 16


In [28]:
truncated_tokens = (
    tokenizer
    .convert_ids_to_tokens(
        truncated_example[
            "input_ids"
        ]
    )
)

truncated_tokens

['[CLS]',
 'this',
 'movie',
 'was',
 'very',
 'interesting',
 '.',
 'this',
 'movie',
 'was',
 'very',
 'interesting',
 '.',
 'this',
 'movie',
 '[SEP]']

In [29]:
encoded_batch = tokenizer(
    batch_texts,
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt",
)

encoded_batch

{'input_ids': tensor([[  101,  2307,  3185,  1012,   102,     0,     0,     0,     0,     0,
             0],
        [  101,  2023,  3185,  2001, 10889,  2204,  1998,  2200, 14036,  1012,
           102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [31]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    "Device:",
    device,
)

Device: cuda


## 3. Load the pretrained DistilBERT classifier

We load pretrained DistilBERT weights and attach a classification
head for our two IMDb sentiment classes.

Classes:

- 0 = negative
- 1 = positive

In [32]:
id2label = {
    0: "neg",
    1: "pos",
}

label2id = {
    "neg": 0,
    "pos": 1,
}

model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        MODEL_NAME,
        num_labels=2,
        id2label=id2label,
        label2id=label2id,
    )
)

model = model.to(
    device
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
print(
    "Number of labels:",
    model.config.num_labels,
)

print(
    "ID to label:",
    model.config.id2label,
)

print(
    "Label to ID:",
    model.config.label2id,
)

Number of labels: 2
ID to label: {0: 'neg', 1: 'pos'}
Label to ID: {'neg': 0, 'pos': 1}


In [34]:
example_batch = tokenizer(
    batch_texts,
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt",
)

example_batch

{'input_ids': tensor([[  101,  2307,  3185,  1012,   102,     0,     0,     0,     0,     0,
             0],
        [  101,  2023,  3185,  2001, 10889,  2204,  1998,  2200, 14036,  1012,
           102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [35]:
example_batch = {
    key: value.to(device)
    for key, value
    in example_batch.items()
}

In [37]:
model.eval()

with torch.inference_mode():
    outputs = model(
        **example_batch
    )

In [38]:
logits = outputs.logits

print(
    "Logits:"
)

print(
    logits
)

print(
    "Shape:",
    logits.shape,
)

Logits:
tensor([[0.0263, 0.0173],
        [0.0706, 0.0154]], device='cuda:0')
Shape: torch.Size([2, 2])


In [39]:
probabilities = torch.softmax(
    logits,
    dim=-1,
)

probabilities

tensor([[0.5023, 0.4977],
        [0.5138, 0.4862]], device='cuda:0')

In [40]:
probability_sums = (
    probabilities
    .sum(
        dim=1
    )
)

probability_sums

tensor([1., 1.], device='cuda:0')

In [41]:
predicted_class_ids = (
    probabilities
    .argmax(
        dim=1
    )
)

predicted_class_ids

tensor([0, 0], device='cuda:0')

In [42]:
predicted_labels = [
    model.config.id2label[
        class_id.item()
    ]
    for class_id
    in predicted_class_ids
]

predicted_labels

['neg', 'neg']

In [43]:
prediction_demo_df = pd.DataFrame(
    {
        "text": batch_texts,
        "negative_probability": (
            probabilities[
                :, 0
            ]
            .cpu()
            .numpy()
        ),
        "positive_probability": (
            probabilities[
                :, 1
            ]
            .cpu()
            .numpy()
        ),
        "prediction": (
            predicted_labels
        ),
    }
)

prediction_demo_df

,text,negative_probability,positive_probability,prediction
0,Great movie.,0.502256,0.497744,neg
1,This movie was surprisingly good and very ente...,0.513807,0.486193,neg
